# UltraFeedback: small midterm trial

**25 prompts × 4 existing answers. Qwen3-0.6B, 1.7B and 4B, thinking off/on. Up to 900 judgments before retries.**
No 8B model is used. There are **six separate run cells**, one per model and thinking mode.

This notebook contains the prepared data and code. No API key or paid inference service is needed.
It uses your Colab GPU compute units. We are evaluating existing answers, not training a model.

1. Select **Runtime → Change runtime type → T4 GPU** (an L4/A100 also works).
2. Run the setup and progress cells, then run only the model/mode cell you want.
3. The final cell downloads the results ZIP for your report.

**After a disconnection:** run setup again with the SAME run name/settings, inspect saved progress,
then run only the unfinished mode cell. Do not restart the full experiment or change the sample size.
Each run cell saves at most 40 new attempts or about 10 minutes (checked between batches).
Re-run that same cell to continue. No keep-alive script is used; Colab can still end a runtime.

The independent 25-prompt sample is 12 ShareGPT, 7 Evol-Instruct and 6 UltraChat prompts.
Your teammate's frozen files were not supplied, so this is a separate, documented H3 pilot.
GPT-4 ratings are quality proxies, not human ground truth. This does not prove causation or model-size effects.

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'transformers==4.57.1', 'accelerate==1.11.0', 'matplotlib', 'pandas'])
import torch
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('GPU memory:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), 'GiB')
else:
    print('CPU session: you can inspect/download saved results. Select a GPU before running a model cell.')

## Unpack the included data and code
You do not need to upload a second file or log into Hugging Face.
The following large cell contains a compressed copy of the 25 examples and the experiment code.

In [ ]:
import base64, io, zipfile, sys, json
from pathlib import Path
PROJECT = Path('/content/dsc261_uf_midterm')
PROJECT.mkdir(parents=True, exist_ok=True)
PACKAGE = '...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PACKAGE))) as package:
    package.extractall(PROJECT)
sys.path.insert(0, str(PROJECT))
# A re-uploaded notebook may share a runtime with the previous version.
for name in list(sys.modules):
    if name == 'src' or name.startswith('src.'):
        del sys.modules[name]
from src.core import read_jsonl, digest
from src.uf_trial import trial_configs, run_one_mode, collect_progress, diagnostic_report, analyze_trial
prompts = read_jsonl(PROJECT / 'data/trial_25.jsonl')
manifest = json.loads((PROJECT / 'trial_manifest.json').read_text())
assert digest(prompts) == manifest['sample_hash']
print('Ready:', len(prompts), 'prompts; 4 existing answers each.')
print('Sources:', manifest['sources'])

## Save location and small-run settings
Leave `N_PROMPTS = 25` for the recommended midterm trial.
For a very small first check, set it to `5` (180 judgments across all three models).
All models see the same selected prompts. 4B uses batch size 1 to conserve GPU memory.
Do not change settings halfway through a run. Google Drive lets you resume after a disconnection.
If you use temporary storage, download the results before the runtime disconnects.

In [ ]:
N_PROMPTS = 25
MODEL_SIZES = ['0.6B', '1.7B', '4B']
BATCH_SIZE = 4
USE_GOOGLE_DRIVE = True
MAX_NEW_CALLS_PER_CELL = 40
MAX_MINUTES_PER_CELL = 10
# Keep this v2 folder name to resume the previous notebook's saved judgments.
RUN_NAME = f"uf_midterm_{N_PROMPTS}_{'-'.join(MODEL_SIZES)}_fp16_batch{BATCH_SIZE}_v2"
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_ROOT = Path('/content/drive/MyDrive/DSC261_UltraFeedback')
else:
    SAVE_ROOT = Path('/content/DSC261_UltraFeedback')
OUT = SAVE_ROOT / RUN_NAME
OUT.mkdir(parents=True, exist_ok=True)
assert 1 <= N_PROMPTS <= 25
prompts = read_jsonl(PROJECT / 'data/trial_25.jsonl')[:N_PROMPTS]
configs = trial_configs(precision='float16', batch_size=BATCH_SIZE, sizes=MODEL_SIZES)
print('Saving to:', OUT)
print('Planned judgments:', len(prompts) * 6 * len(configs), '(plus at most one retry per unreadable verdict)')
print('Off: temperature .7 / top-p .8. On: .6 / .95. Both: top-k 20, min-p 0.')
print('Existing model folders:', [p.name for p in OUT.glob('Qwen3-*') if p.is_dir()])

## Saved progress and failure diagnosis — no model calls
Run this FIRST after reconnecting. It reads per-model logs, including a model interrupted before its
results were copied into the combined file. "Saved" does not mean "valid".
"FINISHED WITH MISSING BRACKETS" means attempts are exhausted, not that the results are usable.

Your earlier log showed 0/25 brackets advancing for 0.6B off, 4/25 for 0.6B on, and 23/25 for 1.7B off.
We need the saved raw answers to determine whether this is formatting, truncation, or another issue.
This update does not change the judging rules or silently reinterpret invalid outputs.

In [ ]:
import pandas as pd
from IPython.display import display, Image, Markdown
def show_progress():
    rows, status = collect_progress(prompts, configs, OUT, export=True)
    display(pd.DataFrame(status))
    return rows, status
rows, status = show_progress()
diagnostic = diagnostic_report(prompts, configs, OUT)
for example in diagnostic['examples']:
    print()
    print(example['config_id'], example['status'], 'attempt', example['attempt'] + 1)
    print('Bracketed labels:', example['bracketed_labels'])
    print('Answer text:', repr(example['content'][:900]))
print('Full diagnostic file saved:', OUT / 'diagnostics.json')

## Run one model and mode at a time
Every prompt has two semifinals and one final. Each match is judged in both answer orders.
The final is built only after the semifinal winners are known.
A split vote is resolved using a fixed seeded answer-identity priority, never using quality ratings.

Thinking is capped at 2,048 generated tokens. Its block is then closed and up to 512 tokens are
allowed for the short explanation and verdict. Forced closures and invalid/truncated answers are logged.
Unreadable verdicts get one fresh retry; missing matches remain missing.

Each cell only runs its named mode. It stops after the chunk limit and saves progress.
Re-run the same cell until its state is FINISHED, then move on. Completed attempts are never repeated.
A completed but invalid judgment is retained as invalid; exhausted retries are not silently restarted.
A batch interrupted before saving may be recomputed. Reloading a model after disconnection is normal;
loading its weights does not repeat previously saved judgments.
Leave `MAX_NEW_CALLS_PER_CELL = 40` and `MAX_MINUTES_PER_CELL = 10` for short sessions.
Use `None` for either limit if you deliberately want an uninterrupted mode run.

### Qwen3-0.6B — thinking off
Run/re-run only this cell to continue this mode.

In [ ]:
CONFIG_ID = 'qwen3-0.6b-off'
if CONFIG_ID in {c['config_id'] for c in configs}:
    rows, status = run_one_mode(prompts, configs, OUT, CONFIG_ID,
        max_new_calls=MAX_NEW_CALLS_PER_CELL,
        max_seconds=None if MAX_MINUTES_PER_CELL is None else 60 * MAX_MINUTES_PER_CELL)
    display(pd.DataFrame(status))
else:
    print('This model is not in MODEL_SIZES; skipped.')

### Qwen3-0.6B — thinking on
Run/re-run only this cell to continue this mode.

In [ ]:
CONFIG_ID = 'qwen3-0.6b-on'
if CONFIG_ID in {c['config_id'] for c in configs}:
    rows, status = run_one_mode(prompts, configs, OUT, CONFIG_ID,
        max_new_calls=MAX_NEW_CALLS_PER_CELL,
        max_seconds=None if MAX_MINUTES_PER_CELL is None else 60 * MAX_MINUTES_PER_CELL)
    display(pd.DataFrame(status))
else:
    print('This model is not in MODEL_SIZES; skipped.')

### Qwen3-1.7B — thinking off
Run/re-run only this cell to continue this mode.

In [ ]:
CONFIG_ID = 'qwen3-1.7b-off'
if CONFIG_ID in {c['config_id'] for c in configs}:
    rows, status = run_one_mode(prompts, configs, OUT, CONFIG_ID,
        max_new_calls=MAX_NEW_CALLS_PER_CELL,
        max_seconds=None if MAX_MINUTES_PER_CELL is None else 60 * MAX_MINUTES_PER_CELL)
    display(pd.DataFrame(status))
else:
    print('This model is not in MODEL_SIZES; skipped.')

### Qwen3-1.7B — thinking on
Run/re-run only this cell to continue this mode.

In [ ]:
CONFIG_ID = 'qwen3-1.7b-on'
if CONFIG_ID in {c['config_id'] for c in configs}:
    rows, status = run_one_mode(prompts, configs, OUT, CONFIG_ID,
        max_new_calls=MAX_NEW_CALLS_PER_CELL,
        max_seconds=None if MAX_MINUTES_PER_CELL is None else 60 * MAX_MINUTES_PER_CELL)
    display(pd.DataFrame(status))
else:
    print('This model is not in MODEL_SIZES; skipped.')

### Qwen3-4B — thinking off
Run/re-run only this cell to continue this mode.

In [ ]:
CONFIG_ID = 'qwen3-4b-off'
if CONFIG_ID in {c['config_id'] for c in configs}:
    rows, status = run_one_mode(prompts, configs, OUT, CONFIG_ID,
        max_new_calls=MAX_NEW_CALLS_PER_CELL,
        max_seconds=None if MAX_MINUTES_PER_CELL is None else 60 * MAX_MINUTES_PER_CELL)
    display(pd.DataFrame(status))
else:
    print('This model is not in MODEL_SIZES; skipped.')

### Qwen3-4B — thinking on
Run/re-run only this cell to continue this mode.

In [ ]:
CONFIG_ID = 'qwen3-4b-on'
if CONFIG_ID in {c['config_id'] for c in configs}:
    rows, status = run_one_mode(prompts, configs, OUT, CONFIG_ID,
        max_new_calls=MAX_NEW_CALLS_PER_CELL,
        max_seconds=None if MAX_MINUTES_PER_CELL is None else 60 * MAX_MINUTES_PER_CELL)
    display(pd.DataFrame(status))
else:
    print('This model is not in MODEL_SIZES; skipped.')

## Download failure diagnostics — run this if many brackets are missing
This downloads the status counts and a few actual failed answers. It does not spend GPU time.
Share this JSON file for diagnosis. The console progress log alone does not show why parsing failed.

In [ ]:
from google.colab import files
diagnostic_report(prompts, configs, OUT)
files.download(str(OUT / 'diagnostics.json'))

## Tables, graphs and report text
- **Selected quality:** GPT-4 proxy rating of the chosen answer; higher is better.
- **Regret:** best available rating minus the chosen answer's rating; lower is better.
- **Format drift:** Markdown presence of the chosen answer minus that of the proxy-best answer.
- **n = 1, 2, 4:** choosing from one, two, or four answers; n=1/2 reuse the same brackets.

The charts use the same completed prompt set across n within each mode.
The separate paired comparison uses only prompts complete in both modes of each model.
A formatting difference here is descriptive; it does not prove formatting caused a quality loss.

In [ ]:
import pandas as pd
from IPython.display import display, Image, Markdown
rows, status = collect_progress(prompts, configs, OUT, export=True)
if any(s['state'] in ['NOT STARTED', 'PENDING'] for s in status):
    print('PARTIAL RUN: some modes are unfinished. Missing coverage is shown; this is not a completed study.')
summary = analyze_trial(prompts, configs, rows, OUT)
table = pd.read_csv(OUT / 'summary.csv')
display(table[table['subset'] == 'all'][['model', 'thinking', 'n', 'complete_prompts',
        'selected_quality', 'regret', 'format_drift', 'best_quality_selected']])
display(pd.read_csv(OUT / 'run_stats.csv'))
if (OUT / 'ultrafeedback_midterm.png').exists():
    display(Image(filename=str(OUT / 'ultrafeedback_midterm.png')))
display(Markdown((OUT / 'MIDTERM_SUMMARY.md').read_text()))

## Download your midterm results
Keep `judge_results.jsonl` as evidence. The ZIP includes the frozen inputs, raw judgments,
model/settings metadata, quality baselines, summary table, graph and report text.
Report actual sample size and completion counts. Do not claim this completes the 500-prompt H3 study.

In [ ]:
import shutil
from google.colab import files
collect_progress(prompts, configs, OUT, export=True)
shutil.copy2(PROJECT / 'trial_manifest.json', OUT / 'source_trial_manifest.json')
archive = shutil.make_archive('/content/' + RUN_NAME + '_results', 'zip', root_dir=OUT)
files.download(archive)